# Lab 3.5 - Amazon SageMaker: Deploying a model

**Educate edition.** Replaces `en_us/3_5-machinelearning.ipynb`.

## Objectives
* Deploy a machine learning model behind an endpoint
* Get a prediction from the deployed model
* Delete the endpoint
* Run a batch transform over the test dataset

**Prerequisite:** run Lab 3.4 first.

> **The single most important cost lesson in this module is here.**
> A real-time endpoint bills **for every hour it exists**, whether or
> not you send it any traffic. Leaving one running overnight is the
> most common way people burn an entire lab budget. Batch transform,
> by contrast, bills only for the minutes it runs and then shuts
> itself down.

## Lab configuration - CHOOSE YOUR TRACK

This notebook runs in one of two modes. Set the flag in the next cell.

| | `USE_MANAGED_SAGEMAKER = False` (**Track B**) | `USE_MANAGED_SAGEMAKER = True` (**Track A**) |
|---|---|---|
| Where training runs | Inside this notebook | A separate managed SageMaker job |
| Extra AWS cost | **$0** | A few cents per job |
| Needs S3 bucket | No | Yes |
| Needs IAM execution role with S3 access | No | Yes |
| Needs `ml.*` training quota | No | **Yes** |
| You learn | The ML concepts | The ML concepts **+ the SageMaker managed workflow** |

**If you are on a $1 budget, or a restricted sandbox account, use
Track B.** It produces the same model and the same numbers. Track A is
what the original AWS Academy lab does, and is worth showing if your
account and budget allow it.

In [ ]:
# ================= LAB CONFIGURATION =================
USE_MANAGED_SAGEMAKER = False    # <-- set True for the managed-job track

# Only used when USE_MANAGED_SAGEMAKER = True.
# These are the smallest instance types that SageMaker supports for each
# role, chosen to keep the cost down.
TRAIN_INSTANCE    = 'ml.m5.large'
ENDPOINT_INSTANCE = 'ml.t2.medium'
TRANSFORM_INSTANCE = 'ml.m5.large'
# =====================================================

import warnings; warnings.simplefilter('ignore')
import pandas as pd, numpy as np, os, json

print('Track:', 'A (managed SageMaker)' if USE_MANAGED_SAGEMAKER
      else 'B (in-notebook, no extra AWS cost)')

## Step 1 - Rebuild the model if needed

As in the original lab, the model is (re)created here as part of setup
so this notebook stands alone.

This works on **both** tracks. On Track B it reloads (or retrains) the
local `xgboost-model.json`. On Track A it reloads `sm_context.json` from
lab 3.4 - and if you ran lab 3.4 on Track B, so that file does not exist,
it runs the managed training job here instead (~4 min, ~$0.01).

In [ ]:
feature_cols = json.load(open('feature_cols.json'))

train = pd.read_csv('train.csv', header=None,
                    names=['target'] + feature_cols)
validation = pd.read_csv('validation.csv', header=None,
                         names=['target'] + feature_cols)
test = pd.read_csv('test.csv', header=None,
                   names=['target'] + feature_cols)

print('train', train.shape, '| validation', validation.shape,
      '| test', test.shape)

if not USE_MANAGED_SAGEMAKER:
    import xgboost as xgb
    if os.path.exists('xgboost-model.json'):
        booster = xgb.Booster()
        booster.load_model('xgboost-model.json')
        print('Loaded existing model from xgboost-model.json')
    else:
        dtrain = xgb.DMatrix(train[feature_cols], label=train['target'])
        dval   = xgb.DMatrix(validation[feature_cols], label=validation['target'])
        params = {'objective':'binary:logistic','eval_metric':'error',
                  'max_depth':5,'eta':0.2,'subsample':0.8,
                  'min_child_weight':6,'gamma':4,'seed':42}
        booster = xgb.train(params, dtrain, num_boost_round=100,
                            evals=[(dval,'validation')],
                            early_stopping_rounds=10, verbose_eval=False)
        booster.save_model('xgboost-model.json')
        print('Trained and saved a fresh model')

else:
    # ---- Track A: make sure lab 3.4's managed-training context exists ----
    # sm_context.json is written by lab 3.4's Track A cell. If you ran lab 3.4
    # on Track B (the default) and then switched to Track A here, that file
    # will not exist. Rather than fail, recreate it now - the same way this
    # notebook retrains the Track B model above if it is missing.
    # Cost: one managed training job, ~4 minutes, ~$0.01.
    if os.path.exists('sm_context.json'):
        ctx = json.load(open('sm_context.json'))
        print('Loaded sm_context.json from lab 3.4')
        print('  training job:', ctx['training_job'])
    else:
        print('sm_context.json not found.')
        print('Lab 3.4 was not run on Track A, so there is no managed model yet.')
        print('Running that training job now (~4 min, ~$0.01)...')
        print()
        import sagemaker
        from sagemaker.inputs import TrainingInput

        session = sagemaker.Session()
        region  = session.boto_region_name
        bucket  = session.default_bucket()
        prefix  = 'mlfoundations/lab3'
        role    = sagemaker.get_execution_role()

        train_uri = session.upload_data('train.csv', bucket=bucket,
                                        key_prefix=f'{prefix}/train')
        val_uri   = session.upload_data('validation.csv', bucket=bucket,
                                        key_prefix=f'{prefix}/validation')
        image_uri = sagemaker.image_uris.retrieve('xgboost', region,
                                                  version='1.7-1')

        estimator = sagemaker.estimator.Estimator(
            image_uri=image_uri,
            role=role,
            instance_count=1,
            instance_type=TRAIN_INSTANCE,
            output_path=f's3://{bucket}/{prefix}/output',
            sagemaker_session=session,
            max_run=1200,                 # hard stop after 20 min - cost guard
        )
        estimator.set_hyperparameters(
            objective='binary:logistic',
            num_round=100,
            max_depth=5,
            eta=0.2,
            subsample=0.8,
            min_child_weight=6,
            gamma=4,
            early_stopping_rounds=10,
        )
        estimator.fit({
            'train':      TrainingInput(train_uri, content_type='csv'),
            'validation': TrainingInput(val_uri,   content_type='csv'),
        })

        ctx = {'bucket': bucket, 'prefix': prefix, 'region': region,
               'model_data': estimator.model_data,
               'training_job': estimator.latest_training_job.name,
               'image_uri': image_uri}
        with open('sm_context.json', 'w') as f:
            json.dump(ctx, f, indent=2)
        print()
        print('Saved sm_context.json - the training instance has already terminated.')

## Step 2 - "Deploy" the model

### Track B - in-process inference

There is no endpoint to create. The model is already loaded in memory
and we call it directly. This is exactly what the container does behind
an endpoint; we have simply removed the network hop and the hourly
bill.

In [ ]:
if not USE_MANAGED_SAGEMAKER:
    import xgboost as xgb

    def predict_proba(frame):
        return booster.predict(xgb.DMatrix(frame[feature_cols]))

    # single-record prediction, the equivalent of one endpoint invocation
    one = test.iloc[[0]]
    p = predict_proba(one)[0]
    print('Features:')
    print(one[feature_cols].to_string(index=False))
    print()
    print(f'Predicted probability of Abnormal: {p:.4f}')
    print('Predicted class:', 'Abnormal' if p > 0.5 else 'Normal')
    print('Actual class   :', 'Abnormal' if one["target"].iloc[0] == 1 else 'Normal')
else:
    print('Track A selected - skip this cell.')

### Track A - deploy a real-time endpoint

This creates a persistent HTTPS endpoint backed by a running instance.

`ml.t2.medium` is the cheapest instance type SageMaker supports for
real-time inference.

**Watch the clock.** The cell after next deletes the endpoint. Do not
skip it.

In [ ]:
if USE_MANAGED_SAGEMAKER:
    import sagemaker
    from sagemaker.serializers import CSVSerializer

    ctx = json.load(open('sm_context.json'))
    session = sagemaker.Session()

    model = sagemaker.model.Model(
        image_uri=ctx['image_uri'],
        model_data=ctx['model_data'],
        role=sagemaker.get_execution_role(),
        sagemaker_session=session,
    )

    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=ENDPOINT_INSTANCE,
        serializer=CSVSerializer(),
    )
    print('Endpoint live:', predictor.endpoint_name)
    print('BILLING NOW. Delete it as soon as you are done.')
else:
    print('Track B selected - skip this cell.')

In [ ]:
if USE_MANAGED_SAGEMAKER:
    row = test[feature_cols].iloc[[0]].to_csv(header=False, index=False).strip()
    result = predictor.predict(row).decode('utf-8')
    p = float(result)
    print('Sent :', row)
    print(f'Predicted probability of Abnormal: {p:.4f}')
    print('Predicted class:', 'Abnormal' if p > 0.5 else 'Normal')
    print('Actual class   :', 'Abnormal' if test["target"].iloc[0] == 1 else 'Normal')
else:
    print('Track B selected - skip this cell.')

## Step 3 - Delete the endpoint

**Run this as soon as you have your prediction.** This is the step
most often forgotten, and it is the difference between a lab that
costs cents and one that costs dollars.

In [ ]:
if USE_MANAGED_SAGEMAKER:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print('Endpoint deleted. Billing stopped.')
else:
    print('Track B - nothing to delete (no endpoint was ever created).')

### Verify nothing is left running

Always confirm. This lists any endpoint still alive in your account.

In [ ]:
try:
    import boto3
    sm = boto3.client('sagemaker')
    eps = sm.list_endpoints()['Endpoints']
    if eps:
        print('WARNING - endpoints still running (still billing):')
        for e in eps:
            print(' ', e['EndpointName'], e['EndpointStatus'], e['CreationTime'])
    else:
        print('No endpoints running. Nothing is billing.')
except Exception as e:
    print('Could not check (no credentials/permissions):', type(e).__name__, e)

## Step 4 - Batch transform

A **batch transform** scores a whole file at once. SageMaker starts an
instance, processes the input, writes results to S3, and terminates the
instance automatically.

Compared with a real-time endpoint:

| | Real-time endpoint | Batch transform |
|---|---|---|
| Billing | Per hour, continuously, until deleted | Only while the job runs |
| Shuts down by itself | **No** | **Yes** |
| Use when | You need low-latency predictions on demand | You have a file of records to score |

For this lab's 31-row test set, batch transform is clearly the right
tool - and the cheaper one.

In [ ]:
if not USE_MANAGED_SAGEMAKER:
    # Track B: the batch equivalent - score the whole test file at once
    probs = predict_proba(test)
    preds = (probs > 0.5).astype(int)

    out = test[feature_cols].copy()
    out['probability'] = probs.round(4)
    out['predicted'] = np.where(preds == 1, 'Abnormal', 'Normal')
    out['actual'] = np.where(test['target'] == 1, 'Abnormal', 'Normal')
    out['correct'] = out['predicted'] == out['actual']

    pd.DataFrame({'probability': probs}).to_csv(
        'test_predictions.csv', header=False, index=False)

    print(f'Scored {len(out)} records')
    print(f'Accuracy on the test set: {out["correct"].mean():.1%}')
    print()
    out[['probability','predicted','actual','correct']].head(10)
else:
    print('Track A selected - skip this cell.')

In [ ]:
if USE_MANAGED_SAGEMAKER:
    import sagemaker

    ctx = json.load(open('sm_context.json'))
    session = sagemaker.Session()
    bucket, prefix = ctx['bucket'], ctx['prefix']

    batch_in = session.upload_data('test_features.csv',
                                   bucket=bucket,
                                   key_prefix=f'{prefix}/batch-in')
    batch_out = f's3://{bucket}/{prefix}/batch-out'
    print('Input :', batch_in)
    print('Output:', batch_out)

    transformer = model.transformer(
        instance_count=1,
        instance_type=TRANSFORM_INSTANCE,
        output_path=batch_out,
        assemble_with='Line',
        accept='text/csv',
    )
    transformer.transform(batch_in, content_type='text/csv', split_type='Line')
    transformer.wait()
    print('Batch transform complete - the instance has already terminated.')

    # pull the results back
    import boto3
    s3 = boto3.client('s3')
    key = f'{prefix}/batch-out/test_features.csv.out'
    s3.download_file(bucket, key, 'test_predictions.csv')
    probs = pd.read_csv('test_predictions.csv', header=None)[0].values

    out = test[feature_cols].copy()
    out['probability'] = probs.round(4)
    out['predicted'] = np.where(probs > 0.5, 'Abnormal', 'Normal')
    out['actual'] = np.where(test['target'] == 1, 'Abnormal', 'Normal')
    out['correct'] = out['predicted'] == out['actual']
    print(f'Accuracy on the test set: {out["correct"].mean():.1%}')
    out[['probability','predicted','actual','correct']].head(10)
else:
    print('Track B selected - skip this cell.')

## Conclusion

You have:
* Deployed a model and obtained a prediction
* Deleted the endpoint and verified nothing is still billing
* Run a batch transform over the test dataset

`test_predictions.csv` is written for Lab 3.6.

**Cost checkpoint - confirm all three:**
1. The endpoint list above is empty
2. No training jobs are in progress
3. **Stop the notebook instance** if you are pausing here

Next: `3_6-machinelearning.ipynb`